# Анализ факторов конверсии в интернет-магазине

Цель: определить поведенческие и сезонные факторы, связанные с покупкой, и подготовить рекомендации для бизнеса.

In [7]:
import io
import zipfile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_URL = 'https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip'

with urlopen(DATA_URL) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
    with archive.open('online_shoppers_intention.csv') as source:
        df = pd.read_csv(source)

df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## Обзор данных

In [8]:
print(f'Количество сессий: {len(df):,}')
print(f'Общая конверсия: {df["Revenue"].mean():.1%}')
df.info()

Количество сессий: 12,330
Общая конверсия: 15.5%
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12

## Конверсия по типу посетителя

In [9]:
conversion_by_visitor = (
    df.groupby('VisitorType', as_index=False)
    .agg(
        Количество_сессий=('Revenue', 'size'),
        Конверсия=('Revenue', 'mean')
    )
    .rename(columns={'VisitorType': 'Тип посетителя'})
)

conversion_by_visitor

ValueError: Length mismatch: Expected axis has 4 elements, new values have 3 elements

## Сезонность

In [ ]:
month_order = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly = df.groupby('Month', as_index=False)['Revenue'].mean()
monthly['Month'] = pd.Categorical(monthly['Month'], categories=month_order, ordered=True)
monthly = monthly.sort_values('Month')

sns.set_theme(style='whitegrid')
plt.figure(figsize=(10, 5))
sns.barplot(data=monthly, x='Month', y='Revenue', color='#4C78A8')
plt.title('Конверсия в покупку по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Конверсия')
plt.ylim(0, 0.3)
plt.show()

## Поведение до покупки

In [ ]:
behavior = df.groupby('Revenue')[['PageValues', 'BounceRates', 'ExitRates']].mean().round(3)
behavior.index = behavior.index.map({False: 'Без покупки', True: 'С покупкой'})
behavior

## Выводы и рекомендации

- Усилить маркетинг перед ноябрьским пиком спроса.
- Проверить воронку возвращающихся посетителей: они составляют основную часть трафика, но конвертируются слабее новых.
- Использовать элементы страниц с высоким PageValues в посадочных страницах.
- Отдельно тестировать офферы в выходные: конверсия там выше.